In [1]:
import pandas as pd
import numpy as np
import copy
import ast
import re 
from pyreadstat import read_sav, metadata_container
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


def transform_with_captured_group(text, capturing_group:str=None, prefix:str="", suffix:str="")->str:
    """
    Extract the first captured group from text and optionally add a prefix
    and/or suffix.

    Parameters
    ----------
    text : str
        Text to search.
    capturing_group : str, optional
        Regular expression containing a capturing group. The first match's
        first capturing group is returned.
    prefix : str, default=""
        Text to prepend to the captured group.
    suffix : str, default=""
        Text to append to the captured group.

    Returns
    -------
    str
        The prefix, captured text, and suffix concatenated.

    Raises
    ------
    AttributeError
        If the pattern does not match or has no capturing group.
    """
    return f"{prefix}{re.search(capturing_group, text).group(1)}{suffix}"

def same_answer_quality_check(dataframe_to_update:pd.DataFrame, column_selection_like:str="", column_selection_regex:str="") -> pd.DataFrame:
    """
    Function that checks strightlining behaviour or same anwer patterns for a series of variables. Useful to detecting quality issues with the data set used in segmentations. 
    
    :param dataframe_to_update: Original Data Frame
    :type dataframe_to_update: pd.DataFrame
    :param column_selection_like: Selecting the columns to analyze with a pd.DataFrame.filter(like="Pattern") function. 
    :type column_selection_like: str
    :param column_selection_regex: Selecting the columns to analyze with a pd.DataFrame.filter(regex=r"Pattern") function. 
    :type column_selection_regex: str
    :return: Dataframe with the added columns of data quality. Uses the filter "like" or "regex" patter as the column name with "_QUALITY_SCORE" suffix as the name. 
    :rtype: DataFrame
    """
    
    updated_frame = dataframe_to_update.copy()

    if column_selection_like != "":
        mode_series = updated_frame.filter(like=column_selection_like).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(like=column_selection_like).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(like=column_selection_like).notna().sum(axis=1)
        updated_frame[f"{column_selection_like}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)

    elif column_selection_regex != "":
        mode_series = updated_frame.filter(regex=column_selection_regex).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(regex=column_selection_regex).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(regex=column_selection_regex).notna().sum(axis=1)
        updated_frame[f"{column_selection_regex}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)
    else:
        print("ERROR: Please supply a filter valid filter string, either as like filter or regex filter.")
    
    return updated_frame

def rename_meta_variables(meta:metadata_container, rename_map: dict, inplace: bool = False)-> metadata_container:
    """
    Return a metadata_container whose variable-name-keyed structures use the new names.

    Parameters
    ----------
    meta : pyreadstat metadata_container
    rename_map : dict  {old_variable_name: new_variable_name}
    inplace : bool     modify the given meta instead of a copy
    """
    new_meta = meta if inplace else copy.deepcopy(meta)

    def _rn(name):
        return rename_map.get(name, name)

    # 1) attributes that are dicts keyed by variable name
    dict_attrs = [
        "column_names_to_labels",
        "variable_value_labels",
        "variable_to_label",
        "variable_measure",
        "variable_display_width",
        "variable_alignment",
        "variable_storage_width",
        "original_variable_types",
        "readstat_variable_types",
        "missing_ranges",
        "missing_user_values",
        "variable_format",
    ]
    for attr in dict_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, dict):
            setattr(new_meta, attr, {_rn(k): v for k, v in old.items()})

    # 2) attributes that are lists of variable names
    list_attrs = ["column_names", "mr_sets"]
    for attr in list_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, list):
            setattr(new_meta, attr, [_rn(v) for v in old])

    # 3) multiple response sets: dict of set_name -> {..., "variable_list": [...]}
    mrsets = getattr(new_meta, "mrset", None) or getattr(new_meta, "multiple_response_sets", None)
    if isinstance(mrsets, dict):
        for set_name, cfg in mrsets.items():
            if isinstance(cfg, dict) and isinstance(cfg.get("variable_list"), list):
                cfg["variable_list"] = [_rn(v) for v in cfg["variable_list"]]

    return new_meta

def dataframe_stacking(df:pd.DataFrame, id_vars_list:list, key_pattern:str, keep_key:bool=True, key_column_name:str="STACK_KEY") -> pd.DataFrame:
    """
    Reshape a DataFrame from wide format to stacked format.

    All columns not listed in ``id_vars_list`` are treated as value columns.
    Their names are transformed using ``key_pattern``. If ``keep_key`` is
    ``True``, the full regex match is stored in a key column; the matching
    text is removed from the column names in either case. Rows with missing
    values are dropped before the result is pivoted back to wide format.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to reshape.
    id_vars_list : list of str
        Columns to retain as identifier variables.
    key_pattern : str
        Regular expression matched against value-column names. The full match
        is used as the key when ``keep_key`` is ``True``.
    keep_key : bool, default=True
        Whether to include the extracted key as an identifier column.
    key_column_name : str, default="STACK_KEY"
        Name of the key column when ``keep_key`` is ``True``.

    Returns
    -------
    pandas.DataFrame
        Reshaped DataFrame with the identifier columns and pivoted values.
    """
    
    stacked_df = df.copy()
    stacked_df = stacked_df.melt(id_vars=id_vars_list)    
    extended_id_vars_list = id_vars_list.copy()
    extended_id_vars_list.append(key_column_name)

    if keep_key:
        stacked_df[key_column_name] = stacked_df['variable'].apply(lambda var: re.match(pattern=key_pattern, string=var)[0])
        
    stacked_df["variable"] = stacked_df["variable"].apply(lambda var: re.sub(pattern=key_pattern, repl="", string=var))
    stacked_df.dropna(subset=["value"], inplace=True)
    
    if keep_key:
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=extended_id_vars_list)
    else: 
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=id_vars_list)
    
    return stacked_df.reset_index()

def parse_stack_column_config(column_name: str) -> dict:
    """
    Parse a stack configuration column name into its embedded Python dictionary.

    The function expects column names produced by the stacking workflow, where
    the configuration is stored as a literal Python object after the
    "STACK_" prefix. For example, a column named
    "STACK_{'source': 'A', 'key': 'Q1'}" is converted back into the
    corresponding dictionary.

    Parameters
    ----------
    column_name : str
        Name of the stack configuration column. It must begin with "STACK_".

    Returns
    -------
    dict
        The dictionary represented by the text that follows the "STACK_" prefix.

    Raises
    ------
    ValueError
        If the column name does not start with the required "STACK_" prefix.
    """
    prefix = "STACK_"

    if not column_name.startswith(prefix):
        raise ValueError(f"Not a stack configuration column: {column_name}")

    config_str = column_name.removeprefix(prefix)

    return ast.literal_eval(config_str)

class ColumnMapper(BaseEstimator, TransformerMixin):

    def __init__(self, mappings):
        self.mappings = mappings

    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
        X = X.copy()

        for col, mapping in self.mappings.items():
            if col in X.columns:
                X[col] = X[col].replace(mapping)

        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_
    
class TopNBoxEncoder(BaseEstimator, TransformerMixin):
    """
    Create cumulative Top-N-Box or Bottom-N-Box binary columns.

    Parameters
    ----------
    n_boxes : dict
        Column names mapped to the number of box columns to create.
        For example, {"A": 2} creates A_T1B and A_T2B when top=True,
        or A_B1B and A_B2B when top=False.

    limit_values : dict or None, default=None
        Theoretical maximum for each column when top=True, or minimum
        when top=False. Unspecified limits are learned from training data.

    handle_missing : {"zero", "nan"}, default="zero"
        Whether missing values produce zeros or remain missing.

    keep_original : bool, default=False
        Whether to include the original columns in the output.

    top : bool, default=True
        If True, encode top boxes; otherwise, encode bottom boxes.
    """

    def __init__(
        self,
        n_boxes,
        limit_values=None,
        handle_missing="zero",
        keep_original=False,
        top=True,
    ):
        self.n_boxes = n_boxes
        self.limit_values = limit_values
        self.handle_missing = handle_missing
        self.keep_original = keep_original
        self.top = top

    def fit(self, X, y=None):
        X = self._to_dataframe(X)

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]

        self._validate_parameters(X)

        # Store the upper or lower limit used for each column.
        self.limit_values_ = {}

        supplied_limit_values = (
            {} if self.limit_values is None else self.limit_values
        )

        for column in self.n_boxes:
            if column in supplied_limit_values:
                limit = supplied_limit_values[column]
            else:
                limit = (
                    X[column].max(skipna=True)
                    if self.top
                    else X[column].min(skipna=True)
                )

            if pd.isna(limit):
                raise ValueError(
                    f"Cannot determine the limit for column {column!r}. "
                    "The column contains only missing values. Provide its "
                    "limit through limit_values."
                )

            self.limit_values_[column] = limit

        self.output_feature_names_ = self._build_feature_names()

        return self

    def transform(self, X):
        check_is_fitted(
            self,
            attributes=[
                "feature_names_in_",
                "limit_values_",
                "output_feature_names_",
            ],
        )

        X = self._to_dataframe(
            X,
            expected_columns=self.feature_names_in_,
        )

        generated_columns = {}
        box_prefix = "T" if self.top else "B"

        if self.keep_original:
            for column in X.columns:
                generated_columns[column] = X[column].copy()

        for column, n_boxes in self.n_boxes.items():
            values = pd.to_numeric(X[column], errors="coerce")
            limit = self.limit_values_[column]

            for n in range(1, n_boxes + 1):
                output_name = f"{column}_{box_prefix}{n}B"

                if self.top:
                    threshold = limit - n + 1
                    encoded = values.ge(threshold) & values.le(limit)
                else:
                    threshold = limit + n - 1
                    encoded = values.le(threshold) & values.ge(limit)

                if self.handle_missing == "nan":
                    encoded = encoded.astype("Float64")
                    encoded = encoded.mask(values.isna(), pd.NA)
                else:
                    encoded = encoded.fillna(False).astype(np.uint8)

                generated_columns[output_name] = encoded

        return pd.DataFrame(
            generated_columns,
            index=X.index,
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, attributes=["output_feature_names_"])
        return np.asarray(self.output_feature_names_, dtype=object)

    def _build_feature_names(self):
        names = []
        box_prefix = "T" if self.top else "B"

        if self.keep_original:
            names.extend(self.feature_names_in_)

        for column, n_boxes in self.n_boxes.items():
            names.extend(
                f"{column}_{box_prefix}{n}B"
                for n in range(1, n_boxes + 1)
            )

        return np.asarray(names, dtype=object)

    def _validate_parameters(self, X):
        if not isinstance(self.n_boxes, dict):
            raise TypeError("n_boxes must be a dictionary.")

        if self.limit_values is not None:
            if not isinstance(self.limit_values, dict):
                raise TypeError("limit_values must be a dictionary or None.")

        if not isinstance(self.top, (bool, np.bool_)):
            raise TypeError("top must be a boolean.")

        if self.handle_missing not in {"zero", "nan"}:
            raise ValueError(
                "handle_missing must be either 'zero' or 'nan'."
            )

        missing_columns = set(self.n_boxes) - set(X.columns)

        if missing_columns:
            raise ValueError(
                "The following columns were not found in X: "
                f"{sorted(missing_columns)}"
            )

        invalid_n_values = {
            column: n
            for column, n in self.n_boxes.items()
            if (
                not isinstance(n, (int, np.integer))
                or isinstance(n, bool)
                or n < 1
            )
        }

        if invalid_n_values:
            raise ValueError(
                "The number of box variables must be a positive "
                f"integer. Invalid values: {invalid_n_values}"
            )

        supplied_limit_values = (
            {} if self.limit_values is None else self.limit_values
        )

        # Retain the original upper-limit check for top boxes only.
        if self.top:
            invalid_boxes = {
                column: n
                for column, n in self.n_boxes.items()
                if (
                    column in supplied_limit_values
                    and n > supplied_limit_values[column]
                )
            }

            if invalid_boxes:
                raise ValueError(
                    "The requested number of boxes cannot be greater than "
                    "the supplied maximum value. Invalid values: "
                    f"{invalid_boxes}"
                )

    @staticmethod
    def _to_dataframe(X, expected_columns=None):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()

            if expected_columns is not None:
                missing = set(expected_columns) - set(X_df.columns)

                if missing:
                    raise ValueError(
                        f"Missing columns during transform: {sorted(missing)}"
                    )

                X_df = X_df.loc[:, expected_columns]

            return X_df

        X_array = np.asarray(X)

        if X_array.ndim != 2:
            raise ValueError("X must be a two-dimensional structure.")

        if expected_columns is None:
            columns = [
                f"x{i}" for i in range(X_array.shape[1])
            ]
        else:
            if X_array.shape[1] != len(expected_columns):
                raise ValueError(
                    "The number of columns during transform does not match "
                    "the number seen during fit."
                )

            columns = expected_columns

        return pd.DataFrame(X_array, columns=columns)


# Step 1: Preprocessing and cleaning of dataset

In [25]:
#### RUN ONLY FOR THE FIRST TIME! 

# Load database 
data_cleaning, meta_cleaning = read_sav("input_database.sav")

# VAR_CLEANING_FRAME preparation 
VAR_CLEANING_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
VAR_CLEANING_FRAME[["N_VAR"]] = ""

# Prepare data cleaning Excel 
with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)

In [26]:
#### RESUME WORK FROM HERE! 

# Loads the data cleaning configuration file. We fill the columns as following: 
# VAR = Original variable name, only keep those we are going to use
# N_VAR = New variable name. Copy original ones and update those needed for preprocessing reasons. 

data_cleaning, meta_cleaning = read_sav("input_database.sav")

VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")

MAIN_VAR_DICT = VAR_CLEANING_FRAME["N_VAR"].to_dict()

# Loads, select variables and renames relevant columns
data_cleaning, meta_cleaning = read_sav("input_database.sav")
data_cleaning = data_cleaning[VAR_CLEANING_FRAME.index] # Selects only relevant variables 
meta_cleaning = rename_meta_variables(meta_cleaning, MAIN_VAR_DICT) # Renames meta
data_cleaning = data_cleaning.rename(columns=MAIN_VAR_DICT) # Renames column names

In [27]:
#### RUN ONLY FOR THE FIRST TIME! 

# PIPE CONFIG FRAME preparation 
STACK_CONFIG_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
STACK_CONFIG_FRAME[["ex_STACK_{\"key_pattern\": \"regex_pattern\", \"keep_key\": True, \"key_column_name\": \"STACK_KEY\"}"]] = ""

with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.reset_index().to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)
    STACK_CONFIG_FRAME.to_excel(writer, sheet_name="STACK_CONFIG_FRAME", index=False)

In [28]:
VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")
STACK_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="STACK_CONFIG_FRAME").set_index("VAR")

# Stacking loop
for col in STACK_CONFIG_FRAME.columns:

    if col.startswith("STACK_"):

        # Parse parameters from column name
        params = parse_stack_column_config(col)

        # Extracting ID vars lists 
        ID_VARS_LIST_ = STACK_CONFIG_FRAME[STACK_CONFIG_FRAME[col]=="ID"].index.to_list()

        data_cleaning = dataframe_stacking(
            df=data_cleaning,
            id_vars_list=ID_VARS_LIST_,
            **params,
        )

In [29]:
#### RUN ONLY FOR THE FIRST TIME! 

# PIPE CONFIG FRAME preparation 
PIPE_CONFIG_FRAME = data_cleaning.columns.to_frame(index=False, name="VAR")
PIPE_CONFIG_FRAME[["DICT_REP","ONE_HOT","TOP_N","TOP_N_MAX", "BOTTOM_N", "BOTTOM_N_MIN", "ex_STACK_{\"key_pattern\": \"regex_pattern\", \"keep_key\": True, \"key_column_name\": \"STACK_KEY\"}"]] = ""

with pd.ExcelWriter("dataframe_configuration.xlsx", engine="openpyxl") as writer:
    VAR_CLEANING_FRAME.reset_index().to_excel(writer, sheet_name="VAR_CLEANING_FRAME", index=False)
    STACK_CONFIG_FRAME.reset_index().to_excel(writer, sheet_name="STACK_CONFIG_FRAME", index=False)
    PIPE_CONFIG_FRAME.to_excel(writer, sheet_name="PIPE_CONFIG_FRAME", index=False)

In [30]:
#### RESUME WORK FROM HERE! 
VAR_CLEANING_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="VAR_CLEANING_FRAME").set_index("VAR")
STACK_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="STACK_CONFIG_FRAME").set_index("VAR")
PIPE_CONFIG_FRAME = pd.read_excel("dataframe_configuration.xlsx", sheet_name="PIPE_CONFIG_FRAME").set_index("VAR")


DICT_REP_COLS = PIPE_CONFIG_FRAME.dropna(subset="DICT_REP").index.to_list()
ONE_HOT_COLS = PIPE_CONFIG_FRAME.dropna(subset="ONE_HOT").index.to_list()
TOP_N_COLS = PIPE_CONFIG_FRAME.dropna(subset="TOP_N").index.to_list()
BOTTOM_N_COLS = PIPE_CONFIG_FRAME.dropna(subset="BOTTOM_N").index.to_list()

DICT_REP_MAPPING = PIPE_CONFIG_FRAME["DICT_REP"].dropna().apply(lambda x: ast.literal_eval(x)).to_dict()
TOP_N_COLS_MAPPING = PIPE_CONFIG_FRAME["TOP_N"].dropna().astype("int").to_dict()
TOP_N_COLS_MAX_MAPPING = PIPE_CONFIG_FRAME["TOP_N_MAX"].dropna().astype("int").to_dict()
BOTTOM_N_COLS_MAPPING = PIPE_CONFIG_FRAME["BOTTOM_N"].dropna().astype("int").to_dict()
BOTTOM_N_COLS_MIN_MAPPING = PIPE_CONFIG_FRAME["BOTTOM_N_MIN"].dropna().astype("int").to_dict()

# Instanciating preprocessing transformer 
preprocessing_trasnformer = ColumnTransformer([
    ("DRT", ColumnMapper(mappings=DICT_REP_MAPPING), DICT_REP_COLS),
    ("OHT", OneHotEncoder(sparse_output=False), ONE_HOT_COLS),
    ("TBT", TopNBoxEncoder(n_boxes=TOP_N_COLS_MAPPING, limit_values=TOP_N_COLS_MAX_MAPPING, handle_missing="zero", keep_original=True, top=True), TOP_N_COLS),
    ("BBT", TopNBoxEncoder(n_boxes=BOTTOM_N_COLS_MAPPING, limit_values=BOTTOM_N_COLS_MIN_MAPPING, handle_missing="zero", keep_original=True, top=False), BOTTOM_N_COLS),
], remainder="passthrough")

# Fit and transform 
preprocessing_trasnformer.fit(data_cleaning)
clean_data = pd.DataFrame(data=preprocessing_trasnformer.transform(data_cleaning), columns=preprocessing_trasnformer.get_feature_names_out())

clean_data.to_csv("clean_data.csv")
clean_data.to_pickle("clean_data.plk")

# Step 2: Preanalysis report